# Factor Weight Optimization — 700 Small/Mid-Cap Stocks**Goal:** Find the optimal % weight for each of the 9 ranking factors(Growth, Upside, Acceleration, Valuation, LT Health, Cash Runway,Conviction, Entry, Momentum) to maximize Pearson correlation withactual forward returns.**Method:**1. Fetch factor scores for ~700 small/mid-cap stocks as of mid-20252. Compute actual returns from mid-2025 to today3. 80/20 cross-validation (5 splits) to find optimal weights4. Optimize per strategy (hold_forever, cycle, catalyst)5. Report best weights and out-of-sample Pearson correlation**Anti-overfitting:** 80/20 CV, L2 regularization, weight smoothing (5% granularity).

In [ ]:
import os, sysREPO_ROOT = os.path.dirname(os.path.abspath('__file__'))if REPO_ROOT not in sys.path:    sys.path.insert(0, REPO_ROOT)from portfolio.smid_data_fetcher import fetch_all, load_csv, FACTOR_NAMES, ADJUSTMENT_NAMESfrom portfolio.smid_optimizer import (    cross_validate, strategy_cv, format_weights_table,    evaluate, round_weights, pearson, spearman)import numpy as npprint("Setup complete.")

## Step 1: Fetch Data from Yahoo FinanceFetches factor scores for ~700 tickers as of mid-2025 (12-month lookback).First run takes ~45-60 min. Results are cached to `output/smid_optimization_data.csv`.Delete the CSV or pass `force_refresh=True` to refetch.

In [ ]:
# Set force_refresh=True to refetch from Yahoo Financerecords = fetch_all(lookback_months=12, force_refresh=False)print(f"\nTotal records: {len(records)}")# Strategy distributionstrats = {}for r in records:    s = r["strategy"]    strats[s] = strats.get(s, 0) + 1print(f"Strategy distribution: {strats}")# Return distributionreturns = [r["actual_return"] for r in records]print(f"\nReturn stats:")print(f"  Mean:   {np.mean(returns):+.1f}%")print(f"  Median: {np.median(returns):+.1f}%")print(f"  Std:    {np.std(returns):.1f}%")print(f"  Min:    {np.min(returns):+.1f}%")print(f"  Max:    {np.max(returns):+.1f}%")

## Step 2: Individual Factor CorrelationsBefore optimizing weights, check which factors individually correlatewith forward returns. This gives a baseline.

In [ ]:
print(f"{'Factor':<14} {'Pearson':>8} {'Spearman':>9}")print("-" * 33)for f in FACTOR_NAMES:    factor_vals = [r[f] for r in records]    ret_vals = [r["actual_return"] for r in records]    p = pearson(factor_vals, ret_vals)    s = spearman(factor_vals, ret_vals)    print(f"{f:<14} {p:+.4f}   {s:+.4f}")# Also check adjustmentsprint()for a in ADJUSTMENT_NAMES:    a_vals = [r[a] for r in records]    ret_vals = [r["actual_return"] for r in records]    p = pearson(a_vals, ret_vals)    s = spearman(a_vals, ret_vals)    print(f"{a:<14} {p:+.4f}   {s:+.4f}")

## Step 3: 80/20 Cross-Validation (All Strategies Pooled)Optimizes weights on 80% of data, evaluates on held-out 20%.Repeated 5 times with different random splits.Final weights are averaged across splits.

In [ ]:
cv_result = cross_validate(records, n_splits=5, test_size=0.2, metric="pearson", seed=42)print(f"\n{'='*60}")print(f"SUMMARY (all strategies pooled)")print(f"{'='*60}")print(f"Mean test Pearson:  {cv_result['mean_test_pearson']:+.4f} (\u00b1{cv_result['std_test_pearson']:.4f})")print(f"Mean train Pearson: {cv_result['mean_train_pearson']:+.4f}")print(f"Overfit gap:        {cv_result['overfit_gap']:+.4f}")print(f"Adj multiplier:     {cv_result['avg_adj_multiplier']:.2f}")print(f"\nOptimal weights:")for f in FACTOR_NAMES:    print(f"  {f:<14} {cv_result['avg_weights'][f]*100:>5.1f}%")

## Step 4: Per-Strategy OptimizationRuns separate CV for each strategy type (hold_forever, cycle, catalyst).Each strategy may have different optimal factor weights.

In [ ]:
strat_results = strategy_cv(records, n_splits=5, metric="pearson")print(format_weights_table(cv_result, strat_results))

## Step 5: Results — Best Weights Per StrategySide-by-side comparison of current weights vs optimized weights.

In [ ]:
from portfolio.ranking import STRATEGY_WEIGHTSprint(f"\n{'='*70}")print(f"CURRENT vs OPTIMIZED WEIGHTS")print(f"{'='*70}")strategies = ["hold_forever", "cycle", "catalyst"]factor_labels = {    "growth": "Growth", "upside": "Upside", "accel": "Acceleration",    "valuation": "Valuation (P/S)", "long_term": "LT Health",    "cash_runway": "Cash Runway", "conviction": "Conviction",    "entry": "Entry", "momentum": "Momentum",}# Headerprint(f"\n{'Factor':<18}", end="")for s in strategies:    print(f" {'Current':>8} {'Optim':>8}", end="")print()print("-" * 72)for f in FACTOR_NAMES:    label = factor_labels.get(f, f)    print(f"{label:<18}", end="")    for s in strategies:        current = STRATEGY_WEIGHTS.get(s, {}).get(f, 0)        sr = strat_results.get(s)        if sr:            optim = sr["avg_weights"].get(f, 0)        else:            optim = cv_result["avg_weights"].get(f, 0)        print(f" {current*100:>7.1f}% {optim*100:>7.1f}%", end="")    print()# Performance comparisonprint(f"\n{'Metric':<18}", end="")for s in strategies:    sr = strat_results.get(s)    if sr:        print(f" {'':>8} {sr['mean_test_pearson']:>+7.4f}", end="")    else:        print(f" {'':>8} {'N/A':>8}", end="")print(f"  <- test Pearson")print(f"\nOverall pooled test Pearson: {cv_result['mean_test_pearson']:+.4f}")

## Step 6: Copy-Paste Block for ranking.pyUse these optimized weights in `portfolio/ranking.py` `STRATEGY_WEIGHTS`.

In [ ]:
print("STRATEGY_WEIGHTS = {")for strat in strategies:    sr = strat_results.get(strat)    w = sr["avg_weights"] if sr else cv_result["avg_weights"]    print(f'    "{strat}": {{')    for i, f in enumerate(FACTOR_NAMES):        comma = "," if i < len(FACTOR_NAMES) - 1 else ","        print(f'        "{f}":{" " * (14 - len(f))}{w[f]:.2f}{comma}')    print("    },")print("}")

## Visualization: Weight Comparison

In [ ]:
try:    import matplotlib    matplotlib.use('Agg')    import matplotlib.pyplot as plt    fig, axes = plt.subplots(1, 3, figsize=(18, 6))    x = np.arange(len(FACTOR_NAMES))    width = 0.35    labels = [factor_labels.get(f, f) for f in FACTOR_NAMES]    for idx, strat in enumerate(strategies):        ax = axes[idx]        current_vals = [STRATEGY_WEIGHTS.get(strat, {}).get(f, 0) * 100 for f in FACTOR_NAMES]        sr = strat_results.get(strat)        if sr:            optim_vals = [sr["avg_weights"].get(f, 0) * 100 for f in FACTOR_NAMES]            test_p = sr["mean_test_pearson"]        else:            optim_vals = [cv_result["avg_weights"].get(f, 0) * 100 for f in FACTOR_NAMES]            test_p = cv_result["mean_test_pearson"]        ax.barh(x - width/2, current_vals, width, label="Current", alpha=0.7)        ax.barh(x + width/2, optim_vals, width, label="Optimized", alpha=0.7)        ax.set_yticks(x)        ax.set_yticklabels(labels)        ax.set_xlabel("Weight %")        ax.set_title(f"{strat}\nTest Pearson: {test_p:+.4f}")        ax.legend()        ax.invert_yaxis()    plt.tight_layout()    plt.savefig("output/weight_comparison.png", dpi=150, bbox_inches="tight")    plt.show()    print("Saved: output/weight_comparison.png")except ImportError:    print("matplotlib not installed, skipping visualization")